# Day 76: Capstone Part 1 - Problem Framing & EDA

## Capstone Brief
This three-part capstone mirrors what a real data science project looks like end to end: Part 1 (today) is problem framing and exploratory data analysis (EDA), Part 2 is building and evaluating a model, and Part 3 is deploying that model so it can actually be used. We'll work with the classic Wine dataset, where the task is to classify a wine sample into one of three cultivars based on its chemical measurements (alcohol content, acidity, color intensity, etc.). The code below loads the dataset as a pandas DataFrame - `as_frame=True` gives us feature columns plus a `target` column together in one table, which is convenient for the exploration we're about to do.</cell id="cell-1">


In [ ]:
from sklearn.datasets import load_wine
import pandas as pd

data = load_wine(as_frame=True)  # loads features + target together as a pandas DataFrame
df = data.frame
print(df.shape)  # (rows, columns) - quick sense of dataset size
print(df.head())  # first few rows to see what the raw data actually looks like

## Define the Problem
Before writing any modeling code, a real project should clearly state what's being predicted, what the possible outcomes are, and why it matters - skipping this step often leads to building a technically correct model that answers the wrong question. Here we're explicit that this is a multi-class classification problem (3 wine cultivars, not a binary yes/no), which immediately tells us what kind of output layer/metric/loss will make sense later (e.g. softmax-style multi-class classification rather than binary classification or regression). Printing `data.target_names` shows the actual class labels so we know what the numeric target values (0, 1, 2) correspond to - a small step that avoids confusion later when interpreting predictions.</cell id="cell-3">


In [ ]:
print('Goal: classify wine into one of 3 cultivars based on chemical properties')
print('Target classes:', data.target_names)  # what the numeric target values (0, 1, 2) actually represent

## Explore the Data
Two of the most important early checks in any EDA are: (1) does the data have the types and completeness you expect, and (2) is the target class balanced or skewed? `df.info()` reports each column's data type and non-null count in one shot, which quickly reveals missing values or unexpected types (e.g. a numeric column accidentally stored as text) that would cause problems later. `df['target'].value_counts()` counts how many samples belong to each class - if one class massively outnumbers the others, that's a class imbalance problem that affects both how you should evaluate the model (accuracy alone becomes misleading) and possibly how you train it (e.g. class weighting). Catching these issues now, before modeling, saves a lot of confusion later.</cell id="cell-5">


In [ ]:
print(df.info())  # column dtypes and non-null counts - flags missing values or wrong types
print(df['target'].value_counts())  # count of samples per wine class - checks for class imbalance

## Visual Exploration
Numbers alone (like `.info()` or `.value_counts()`) only tell part of the story - plots make it much easier to spot patterns, outliers, and relationships between features and the target. A boxplot of a feature grouped by class (alcohol content by wine class, below) shows whether that feature actually differs meaningfully between classes - if the boxes barely overlap, that feature is likely a strong predictor; if they overlap heavily, it may be less useful on its own. A correlation heatmap across all features shows which pairs of features move together (values near 1 or -1) versus which are independent (values near 0) - highly correlated features carry redundant information, which is useful to know before feature selection or when interpreting a model's coefficients/importances later.</cell id="cell-7">


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x='target', y='alcohol', data=df)  # compares alcohol content distribution across the 3 wine classes
plt.title('Alcohol Content by Wine Class')
plt.show()

corr = df.drop(columns=['target']).corr()  # pairwise correlation between features only (exclude the target column)
sns.heatmap(corr, cmap='coolwarm')  # red/blue extremes show strong positive/negative correlation
plt.title('Feature Correlation Heatmap')
plt.show()